# Wrapper Stack

This notebook shows that `make_env(...)` is a convenience around a concrete wrapper stack. We will build the same `colour_grid_world` CMDP environment two ways: once with `make_env`, and once manually with each wrapper in order.

The matching static docs page is at `docs/Tutorials/Basics/Wrapper Stack.md`.

## CPU-first setup

The wrapper stack itself is not hardware-sensitive, but using the same CPU-first setup keeps these tutorials consistent and quiet.

In [15]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

from pprint import pprint

ACTION_NAMES = {
    0: "left",
    1: "right",
    2: "down",
    3: "up",
    4: "stay",
}


## Imports

The manual path uses the same pieces that `make_env` applies internally: a base environment, a time limit, labels, a constraint wrapper, and two monitor wrappers.

In [16]:
from pathlib import Path
from shutil import rmtree

from gymnasium.wrappers import RecordVideo

from masa.plugins.helpers import load_plugins
from masa.common.constraints.cmdp import CumulativeCostEnv
from masa.common.labelled_env import LabelledEnv
from masa.common.utils import make_env
from masa.common.wrappers import (
    ConstraintMonitor,
    RewardMonitor,
    TimeLimit,
    get_wrapped,
    is_wrapped,
)
from masa.envs.tabular.colour_grid_world import ColourGridWorld, cost_fn, label_fn

load_plugins()


## Build with `make_env`

`make_env` looks up the environment and constraint by registry name, constructs the base environment, and applies MASA's standard wrapper order.

In [17]:
def build_factory_env():
    return make_env(
        "colour_grid_world",
        "cmdp",
        5,
        label_fn=label_fn,
        cost_fn=cost_fn,
        budget=0.0,
    )

factory_env = build_factory_env()
print(type(factory_env).__name__)
factory_env.close()


RewardMonitor


## Build the same stack manually

The equivalent manual stack is:

```text
ColourGridWorld
-> TimeLimit
-> LabelledEnv
-> CumulativeCostEnv
-> ConstraintMonitor
-> RewardMonitor
```

In [18]:
def build_manual_env():
    env = ColourGridWorld()
    env = TimeLimit(env, 5)
    env = LabelledEnv(env, label_fn)
    env = CumulativeCostEnv(env, cost_fn=cost_fn, budget=0.0)
    env = ConstraintMonitor(env)
    env = RewardMonitor(env)
    return env

manual_env = build_manual_env()
print(type(manual_env).__name__)
manual_env.close()


RewardMonitor


## Inspect the wrapper chain

`is_wrapped` answers whether a wrapper appears anywhere in the chain. `get_wrapped` returns the first matching wrapper object.

In [19]:
WRAPPERS = (TimeLimit, LabelledEnv, CumulativeCostEnv, ConstraintMonitor, RewardMonitor)

def summarize_wrappers(env):
    return {
        wrapper.__name__: {
            "present": is_wrapped(env, wrapper),
            "found_type": type(get_wrapped(env, wrapper)).__name__,
        }
        for wrapper in WRAPPERS
    }

factory_env = build_factory_env()
manual_env = build_manual_env()

factory_summary = summarize_wrappers(factory_env)
manual_summary = summarize_wrappers(manual_env)

print("factory stack")
pprint(factory_summary)
print("manual stack")
pprint(manual_summary)

assert factory_summary == manual_summary

factory_env.close()
manual_env.close()


factory stack
{'ConstraintMonitor': {'found_type': 'ConstraintMonitor', 'present': True},
 'CumulativeCostEnv': {'found_type': 'CumulativeCostEnv', 'present': True},
 'LabelledEnv': {'found_type': 'LabelledEnv', 'present': True},
 'RewardMonitor': {'found_type': 'RewardMonitor', 'present': True},
 'TimeLimit': {'found_type': 'TimeLimit', 'present': True}}
manual stack
{'ConstraintMonitor': {'found_type': 'ConstraintMonitor', 'present': True},
 'CumulativeCostEnv': {'found_type': 'CumulativeCostEnv', 'present': True},
 'LabelledEnv': {'found_type': 'LabelledEnv', 'present': True},
 'RewardMonitor': {'found_type': 'RewardMonitor', 'present': True},
 'TimeLimit': {'found_type': 'TimeLimit', 'present': True}}


## Compare behaviour

Both environments should emit the same observations, rewards, labels, constraint metrics, and done flags for the same seed and actions.

In [20]:
def rollout(build_env, actions, *, seed):
    env = build_env()
    obs, info = env.reset(seed=seed)
    rows = [
        {
            "event": "reset",
            "obs": int(obs),
            "labels": sorted(info["labels"]),
            "constraint": info["constraint"],
        }
    ]

    for step, action in enumerate(actions, start=1):
        obs, reward, terminated, truncated, info = env.step(action)
        rows.append(
            {
                "event": f"step_{step}",
                "action": ACTION_NAMES[action],
                "obs": int(obs),
                "reward": float(reward),
                "terminated": bool(terminated),
                "truncated": bool(truncated),
                "labels": sorted(info["labels"]),
                "constraint": info["constraint"],
                "metrics": info.get("metrics"),
            }
        )
        if terminated or truncated:
            break

    env.close()
    return rows

actions = [2, 2, 2, 2]
factory_rows = rollout(build_factory_env, actions, seed=1)
manual_rows = rollout(build_manual_env, actions, seed=1)

print("factory rollout")
pprint(factory_rows)
print("manual rollout")
pprint(manual_rows)

assert factory_rows == manual_rows
assert factory_rows[-1]["labels"] == ["blue"]
assert factory_rows[-1]["constraint"]["step"]["cost"] == 1.0
assert factory_rows[-1]["constraint"]["step"]["violation"] == 1.0


factory rollout
[{'constraint': {'step': {'cost': 0.0, 'cum_cost': 0.0, 'violation': 0.0},
                 'type': 'cmdp'},
  'event': 'reset',
  'labels': ['start'],
  'obs': 0},
 {'action': 'down',
  'constraint': {'episode': {'cum_cost': 0.0, 'satisfied': 1.0},
                 'step': {'cost': 0.0, 'cum_cost': 0.0, 'violation': 0.0},
                 'type': 'cmdp'},
  'event': 'step_1',
  'labels': [],
  'metrics': {'step': {'reward': 0.0}},
  'obs': 9,
  'reward': 0.0,
  'terminated': False,
  'truncated': False},
 {'action': 'down',
  'constraint': {'episode': {'cum_cost': 0.0, 'satisfied': 1.0},
                 'step': {'cost': 0.0, 'cum_cost': 0.0, 'violation': 0.0},
                 'type': 'cmdp'},
  'event': 'step_2',
  'labels': [],
  'metrics': {'step': {'reward': 0.0}},
  'obs': 18,
  'reward': 0.0,
  'terminated': False,
  'truncated': False},
 {'action': 'down',
  'constraint': {'episode': {'cum_cost': 0.0, 'satisfied': 1.0},
                 'step': {'cost': 0.0, 'c

## Record the finished stack

`RecordVideo` is not part of the semantic MASA stack. When `record_video=True`, `make_env` wraps the completed stack with Gymnasium's video recorder, so labels, constraints, and monitors behave the same while frames are saved from `render()`.

`video_dir = Path("videos/tutorial_wrapper_stack")` stores recordings under the repo-local `videos/` directory when the notebook is run from the project root. The cell prints the exact directory and MP4 paths, and clears this tutorial subdirectory before recording so reruns do not mix old and new videos.

`record_video_episode_trigger` is Gymnasium's `episode_trigger`. It receives the zero-based episode id and records that episode when it returns `True`. Common schedules are:

```python
record_every_episode = lambda episode_id: True
record_every_5_episodes_from_zero = lambda episode_id: episode_id % 5 == 0
```

which can then be passed into
```python
env = make_env(
    ...,
    record_video=True,
    record_video_episode_trigger=trigger,
)
```

Gymnasium's `step_trigger` is useful for fixed-length clips that start immediately on a global environment step. This starts a 500-frame clip every 500 environment steps:

```python
video_kwargs={
    "step_trigger": lambda step_id: step_id > 0 and step_id % 500 == 0,
    "video_length": 500,
}
```

If you specifically want the next complete episode after each 500-step boundary, use a small stateful episode trigger and update it from your rollout loop. The example script exposes this as `--trigger-mode step --trigger-value 500`.

```python
class RecordNextEpisodeEveryNSteps:
    def __init__(self, interval):
        self.interval = interval
        self.total_steps = 0
        self.next_threshold = interval
        self.pending_recordings = 0

    def observe_step(self):
        self.total_steps += 1
        while self.total_steps >= self.next_threshold:
            self.pending_recordings += 1
            self.next_threshold += self.interval

    def __call__(self, episode_id):
        if self.pending_recordings < 1:
            return False
        self.pending_recordings -= 1
        return True

trigger = RecordNextEpisodeEveryNSteps(500)

env = make_env(
    ...,
    record_video=True,
    record_video_episode_trigger=trigger,
)

# Inside the rollout loop, after each env.step(...):
trigger.observe_step()
```

In [21]:
video_dir = Path("videos/tutorial_wrapper_stack")
rmtree(video_dir, ignore_errors=True)
print("video directory", video_dir)

video_env = make_env(
    "colour_grid_world",
    "cmdp",
    len(actions),
    label_fn=label_fn,
    cost_fn=cost_fn,
    budget=0.0,
    env_kwargs={
        "render_mode": "rgb_array",
        "render_window_size": 96,
    },
    record_video=True,
    record_video_episode_trigger=lambda episode_id: True,
    video_folder=str(video_dir),
)

print(type(video_env).__name__)
assert isinstance(video_env, RecordVideo)

try:
    video_env.reset(seed=2)
    for action in actions:
        _, _, terminated, truncated, _ = video_env.step(action)
        if terminated or truncated:
            break
finally:
    video_env.close()

recorded_videos = sorted(video_dir.glob("*.mp4"))
print("recorded videos")
for path in recorded_videos:
    print(path)

assert recorded_videos
assert all(path.stat().st_size > 0 for path in recorded_videos)


video directory videos/tutorial_wrapper_stack
RecordVideo
recorded videos
videos/tutorial_wrapper_stack/rl-video-episode-0.mp4


## Why order matters

- `TimeLimit` comes first so truncation is part of the base interaction before safety monitoring.
- `LabelledEnv` must run before the constraint wrapper because constraints consume `info["labels"]`.
- `CumulativeCostEnv` updates the stateful safety monitor.
- `ConstraintMonitor` reads the constraint and writes `info["constraint"]`.
- `RewardMonitor` is last here so it can add reward and episode-length metrics without changing safety logic.

Most users should call `make_env`. Manual construction is useful when you need to understand, debug, or extend the stack.